In [2]:
from pathlib import Path
import tensorflow as tf
import sys

# appending llm_components path to sys.path to easily import
axiom_utils = Path('/kaggle/input/datasets/harshit1234g/axiomlm-utils')
sys.path.append(str(axiom_utils))
import llm_components as lc

In [3]:
tf.config.list_physical_devices('GPU')

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'),
 PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]

## Paths

In [4]:
dolly_dir = axiom_utils / 'dolly_15k'
features_path = str(dolly_dir / 'processed_features.npy')
labels_path = str(dolly_dir / 'processed_labels.npy')
tokenizer_path = str(axiom_utils / 'sp_tokenizer.model')

# I uploaded the model on kaggle, so it has a different path
model_path = Path('/kaggle/input/models/harshit1234g/axiomlm/tensorflow2/default/4/AxiomLM-33M-Base.keras')

## Loading data

In [5]:
tokenizer = lc.load_sp_tokenizer(tokenizer_path)
pad_id = tokenizer.pad_id()

In [6]:
full_ds = lc.load_sft_dataset(
    features_path,
    labels_path,
    pad_token_id= pad_id,
    batch_size= 64,
    shuffle_buffer= 10_000
)

I0000 00:00:1772724884.735840      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1772724884.738637      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [7]:
batches = 0
for _ in full_ds.as_numpy_iterator():
    batches += 1

In [8]:
train_size = int(0.8 * batches)
val_size = int(0.1 * batches)
test_size = batches - train_size - val_size

In [9]:
print(f'Total batches: {batches}')
print(f'{train_size = }, {val_size = }, {test_size = }')

Total batches: 216
train_size = 172, val_size = 21, test_size = 23


In [10]:
train_ds = full_ds.take(train_size)
val_ds = full_ds.skip(train_size).take(val_size)
test_ds = full_ds.skip(train_size + val_size)

In [11]:
for item in train_ds.take(1):
    print(item[0][0])
    print(item[1][0])

tf.Tensor(
[    1    39  5109  1622 15984    14 15947   286   656   360   312  8210
    67    14    39   363  1298  2898 15984    14 15951  1131   474   360
  2108   293  3248   264  5879 15922   285  2390 10459   347   386  7380
  3428  7327   325   262  1328  5152   913 15935   501   346   842   687
   749  6274   285  5581  1599   359   262   494  1772   607  1325  3483
   292 12783   995  5409   753  3248 15936  4027 15935  1366  1599  3726
   264  1074   285 11434   292   264 10547   285  3233 15935  5808   656
  1009   360   312  8210   285   269  2248 15936   346 11047   293   360
   264 10042   396 15935   737   762  2196 15935   292   737 12783   995
 11765  1599 15952   707  1230   979  3296   262  4506  2089  2275   293
  2239  5727 15952 15922 15589   380 15936     2     3     3     3     3
     3     3     3     3     3     3     3     3     3     3     3     3
     3     3     3     3     3     3     3     3     3     3     3     3
     3     3     3     3     3     3    

## SFT

In [12]:
strategy = tf.distribute.MirroredStrategy()

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')


In [13]:
with strategy.scope():
    model = tf.keras.models.load_model(model_path)
    
    # freezing the initial embedding and transformer layers
    for layer in model.layers[:6]:
        layer.trainable = False

    for layer in model.layers[6:]:
        layer.trainable = True
    
    optimizer = tf.keras.optimizers.AdamW(
        learning_rate= 2e-5,
        weight_decay= 0.0,   # removing regularization, so that model could adapt the new behaviour easily
        clipnorm= 1.0
    )

    loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(
        from_logits= True,
        ignore_class= -100
    )

    model.compile(
        optimizer= optimizer,
        loss= loss_fn,
        metrics= [lc.Perplexity(pad_id= pad_id)]
    )

In [14]:
for layer in model.layers:
    print(f'{layer}: {layer.trainable}')

<Embedding name=embedding, built=True>: False
<Embedding name=embedding_1, built=True>: False
<TransformerBlock name=transformer_block, built=True>: False
<TransformerBlock name=transformer_block_1, built=True>: False
<TransformerBlock name=transformer_block_2, built=True>: False
<TransformerBlock name=transformer_block_3, built=True>: False
<TransformerBlock name=transformer_block_4, built=True>: True
<TransformerBlock name=transformer_block_5, built=True>: True
<TransformerBlock name=transformer_block_6, built=True>: True
<TransformerBlock name=transformer_block_7, built=True>: True
<LayerNormalization name=layer_normalization_16, built=True>: True


In [15]:
model.summary()

Model: "gpt"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (1, 512, 512)          │     8,192,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_1 (Embedding)         │ (512, 512)             │       262,144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block               │ ?                      │     3,150,848 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_1             │ ?                      │     3,150,848 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_2             │ ?                      │     3,150,848 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_3             │ ?                      │     3,150,848 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_4             │ ?                      │     3,150,848 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_5             │ ?                      │     3,150,848 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_6             │ ?                      │     3,150,848 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_7             │ ?                      │     3,150,848 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer_normalization_16          │ ?                      │         1,024 │
│ (LayerNormalization)            │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 33,661,952 (128.41 MB)

 Trainable params: 12,604,416 (48.08 MB)

 Non-trainable params: 21,057,536 (80.33 MB)

In [16]:
history = model.fit(
    train_ds,
    epochs= 3,
    validation_data= val_ds
)

INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Redu

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:160: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


172/172 ━━━━━━━━━━━━━━━━━━━━ 232s 1s/step - loss: 4.8242 - ppl: nan - val_loss: 4.0029 - val_ppl: nan
Epoch 2/3
172/172 ━━━━━━━━━━━━━━━━━━━━ 214s 1s/step - loss: 3.9494 - ppl: nan - val_loss: 3.8557 - val_ppl: nan
Epoch 3/3
172/172 ━━━━━━━━━━━━━━━━━━━━ 212s 1s/step - loss: 3.7798 - ppl: nan - val_loss: 3.7088 - val_ppl: nan


In [17]:
test_loss, test_ppl = model.evaluate(test_ds)
print(f'{test_loss = }\n{test_ppl = }')

23/23 ━━━━━━━━━━━━━━━━━━━━ 18s 639ms/step - loss: 3.7399 - ppl: nan
test_loss = 3.714684009552002
test_ppl = nan


In [18]:
wiki_test = lc.create_dataset_from_npy(
    npy_path= axiom_utils / 'wikitext_npy/test.npy',
    seq_len= 512,
    batch_size= 64,
    shift= 512,
    shuffle_buffer= None,
    training= False
)

In [19]:
wiki_loss, wiki_ppl = model.evaluate(wiki_test)
print(f'{wiki_loss = }\n{wiki_ppl = }')

8/8 ━━━━━━━━━━━━━━━━━━━━ 6s 714ms/step - loss: 3.1866 - ppl: 24.3430
wiki_loss = 3.2217631340026855
wiki_ppl = 25.07228660583496


In [20]:
model.save('AxiomLM-33M-Instruct.keras')